Before running this notbook, please run codonaligner.ipynb and filting_introns.ipynb

In [1]:
import cogent3
from cogent3 import get_app
from cogent3.maths.matrix_exponential_integration import expected_number_subs
import matplotlib.pyplot as plt
import paths
import libs
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm
import os

import trinuc_models as trinucs # this module must be in the same directory as this notebook

# Non cds

In [6]:
#Setting up the rules for model fitting of noncds regions
sm_noncds=trinucs.GT_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_noncds = [{"par_name": n, "is_independent": True, "upper": 100} for n in paramnames if n!="omega"] + [{"par_name": "omega", "value": 1.0, "is_constant": True}]
GT_subsmodel = get_app("model", "GT_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      opt_args={"max_restarts": 20, "tolerance": 1e-8})

#Setting up the app for reading noncds sequences
#rename_noncds eliminates files that do not contain Orangutan, Human and Chimp sequences or that has duplicates.
loader = get_app("load_aligned", moltype="dna")
rename_noncds = libs.renamer_noncds_aligned()
#Need to omit gapped positions. My alignmetns is a subset of 10 primate alignments.
#If a indel spread across all 10 species, then the position is not informative and can be omitted.
omit_gap_pos_app = get_app("omit_gap_pos", moltype="dna")
#Because I am using a trinucleotide I need to omit degenerates using motif 3
omit_degs_noncds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

noncds_app = loader + rename_noncds + omit_gap_pos_app + omit_degs_noncds

## Intergenic Ancestral Repeats (IGAR)

In [3]:
folder_in = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/chrm10/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGAR10 = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
alns_IGAR10 = concat(nonconcat_IGAR10)

folder_in = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGAR22 = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
alns_IGAR22 = concat(nonconcat_IGAR22)

alns_IGAR = concat([alns_IGAR10, alns_IGAR22])
print("alignment length:", len(alns_IGAR))
alns_IGAR.source = "IGAR_alignments"


   0%|          |00:00<?

   0%|          |00:00<?

alignment length: 88026


In [4]:

folder_out = paths.DATA_HUMCHIMPORANG115 + 'intergenicAR/trinucleotide_filtered_chrm22_10/'
out_dstore = cogent3.open_data_store(folder_out, suffix='fa', mode='r')

# 3. Create the writer app
write_seqs_app = get_app("write_seqs", data_store=out_dstore, format_name="fasta")

# 4. Write the alignment
_ = write_seqs_app(alns_IGAR)

In [7]:
result_IGAR = GT_subsmodel(alns_IGAR)

result_IGAR.lf

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -341025.7814
number of free parameters = 79
=======================================================================
(CG>TG | CG>CA)     A>C     A>G     A>T     C>A     C>G     C>T     G>A
-----------------------------------------------------------------------
          15.56    0.99    0.84    1.15    0.89    0.89    0.83    0.78
-----------------------------------------------------------------------

continued: 
=====================================
 G>C     G>T     T>A     T>C    omega
-------------------------------------
0.96    0.86    1.04    0.90     0.96
-------------------------------------

==============================
edge          parent    length
------------------------------
Orangutan     root        3.25
Chimpanzee    root        1.63
Human         root        1.43
------------------------------
============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.05    0.01    0.02    0.03    0.01    0.01    0.01    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.02    0.01    0.01    0.01    0.02    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.02    0.02    0.02    0.01    0.01    0.02    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.02    0.01    0.01    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.01    0.01    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.01    0.02    0.01    0.02    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.02    0.01    0.05
----------------------------